# SupportOpsAI — QLoRA Fine-Tuning

Fine-tuning **Qwen/Qwen2.5-1.5B-Instruct** with 4-bit QLoRA for customer-support ticket classification.

**Task:** predict one support queue and one priority from a ticket subject/body, returning structured JSON.

> **Notebook status:** This notebook is a cleaned presentation copy of the executed experiment. Existing cell outputs are preserved so the recorded results can be reviewed without rerunning the full training/evaluation pipeline.


## 1. Dataset and preprocessing

The experiment uses the processed English customer-support ticket dataset stored in Google Drive. Exact duplicate tickets were removed before splitting. The final dataset contains **23,747 unique English tickets** after removing the single row with a missing body.

The split was performed before fine-tuning using an 80/10/10 train/validation/test split with stratification on the combined queue + priority label.


In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd

clean_path = "/content/drive/MyDrive/SupportOpsAI/data/processed/english_clean.csv"

final_df = pd.read_csv(clean_path)

print(final_df.shape)
print(final_df.columns)

(23747, 6)
Index(['subject', 'body', 'queue', 'priority', 'language', 'version'], dtype='object')


In [3]:
from sklearn.model_selection import train_test_split

final_df["stratify_label"] = (
    final_df["queue"] + " | " + final_df["priority"]
)

train_df, temp_df = train_test_split(
    final_df,
    test_size=0.20,
    random_state=42,
    stratify=final_df["stratify_label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["stratify_label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 18997
Validation: 2375
Test: 2375


In [4]:
train_df = train_df.drop(columns=["stratify_label"])
val_df = val_df.drop(columns=["stratify_label"])
test_df = test_df.drop(columns=["stratify_label"])

In [11]:
train_df.to_csv(
    "/content/drive/MyDrive/SupportOpsAI/data/processed/train.csv",
    index=False
)

val_df.to_csv(
    "/content/drive/MyDrive/SupportOpsAI/data/processed/validation.csv",
    index=False
)

test_df.to_csv(
    "/content/drive/MyDrive/SupportOpsAI/data/processed/test.csv",
    index=False
)

print("Saved train/validation/test splits.")

Saved train/validation/test splits.


In [5]:
def format_ticket(row):
    subject = row["subject"] if row["subject"] else ""
    body = row["body"]

    return {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are a customer support ticket classifier. "
                    "Classify the ticket and return only valid JSON."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Subject: {subject}\n\n"
                    f"Body: {body}"
                )
            },
            {
                "role": "assistant",
                "content": (
                    f'{{"queue": "{row["queue"]}", '
                    f'"priority": "{row["priority"]}"}}'
                )
            }
        ]
    }

In [6]:
train_formatted = train_df.apply(format_ticket, axis=1).tolist()
val_formatted = val_df.apply(format_ticket, axis=1).tolist()
test_formatted = test_df.apply(format_ticket, axis=1).tolist()

print("Train examples:", len(train_formatted))
print("Validation examples:", len(val_formatted))
print("Test examples:", len(test_formatted))

Train examples: 18997
Validation examples: 2375
Test examples: 2375


## 2. Tokenization and chat formatting

Each ticket is formatted as a three-message chat example: system instructions, the ticket as the user message, and the target JSON as the assistant message. Qwen's chat template is then applied before supervised fine-tuning.


In [8]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded


In [12]:
def apply_chat_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return example


train_dataset = train_dataset.map(apply_chat_template)
val_dataset = val_dataset.map(apply_chat_template)
test_dataset = test_dataset.map(apply_chat_template)

Map:   0%|          | 0/18997 [00:00<?, ? examples/s]

Map:   0%|          | 0/2375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2375 [00:00<?, ? examples/s]

In [19]:
print(train_dataset.column_names)

['messages', 'text']


In [20]:
print(train_dataset[0]["text"])

<|im_start|>system
You are a customer support ticket classifier. Classify the ticket and return only valid JSON.<|im_end|>
<|im_start|>user
Subject: Inquiry on Jenkins Integration with Our SaaS Platform

Body: Hello, I am contacting you to request information about integrating Jenkins with your project management software. Could you share any documentation or guides you have for this process? I am particularly interested in understanding the benefits and challenges of this integration. I look forward to your response and to discussing this further.<|im_end|>
<|im_start|>assistant
{"queue": "Product Support", "priority": "medium"}<|im_end|>



In [23]:
token_lengths = [
    len(tokenizer(x["text"])["input_ids"])
    for x in train_dataset
]

print("Min:", min(token_lengths))
print("Max:", max(token_lengths))
print("Average:", sum(token_lengths) / len(token_lengths))

Min: 54
Max: 370
Average: 122.63178396588935


## 3. Base-model baseline

Before fine-tuning, the base Qwen model was evaluated on the untouched test set. These recorded outputs are retained below as the baseline for comparison.


In [13]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded


In [19]:
import torch

SYSTEM_PROMPT = """You are a customer support ticket classifier.

Choose exactly one queue from:
Technical Support
Product Support
Customer Service
IT Support
Billing and Payments
Returns and Exchanges
Service Outages and Maintenance
Sales and Pre-Sales
Human Resources
General Inquiry

Choose exactly one priority from:
low
medium
high

Return ONLY valid JSON in exactly this format:
{"queue": "<queue>", "priority": "<priority>"}
"""

def predict_ticket(subject, body):
    subject = subject if pd.notna(subject) else ""

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Subject: {subject}\n\nBody: {body}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

In [22]:
import json
import re
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm

# Allowed labels
ALLOWED_QUEUES = {
    "Technical Support",
    "Product Support",
    "Customer Service",
    "IT Support",
    "Billing and Payments",
    "Returns and Exchanges",
    "Service Outages and Maintenance",
    "Sales and Pre-Sales",
    "Human Resources",
    "General Inquiry"
}

ALLOWED_PRIORITIES = {"low", "medium", "high"}

# Make sure tokenizer can pad
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"


def parse_prediction(response):
    """
    Extract JSON from model response and validate labels.
    """
    response = response.strip()

    # First try the entire response
    try:
        data = json.loads(response)
    except json.JSONDecodeError:
        # Try extracting a JSON object from surrounding text
        match = re.search(r'\{.*?\}', response, re.DOTALL)

        if not match:
            return None

        try:
            data = json.loads(match.group())
        except json.JSONDecodeError:
            return None

    if not isinstance(data, dict):
        return None

    queue = data.get("queue")
    priority = data.get("priority")

    if queue not in ALLOWED_QUEUES:
        return None

    if priority not in ALLOWED_PRIORITIES:
        return None

    return {
        "queue": queue,
        "priority": priority
    }


def predict_ticket(subject, body):
    subject = subject if pd.notna(subject) else ""

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Subject: {subject}\n\nBody: {body}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


# Run baseline
baseline_results = []

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Running baseline"
):
    raw_output = predict_ticket(
        row["subject"],
        row["body"]
    )

    prediction = parse_prediction(raw_output)

    baseline_results.append({
        "subject": row["subject"],
        "body": row["body"],
        "true_queue": row["queue"],
        "true_priority": row["priority"],
        "raw_output": raw_output,
        "pred_queue": prediction["queue"] if prediction else None,
        "pred_priority": prediction["priority"] if prediction else None,
        "valid_json": prediction is not None
    })

baseline_df = pd.DataFrame(baseline_results)

print("\nBaseline evaluation complete!")
print("Tickets evaluated:", len(baseline_df))

Running baseline:   0%|          | 0/2375 [00:00<?, ?it/s]


Baseline evaluation complete!
Tickets evaluated: 2375


In [24]:
# Only evaluate classification metrics on valid predictions
valid_df = baseline_df[baseline_df["valid_json"]].copy()

print("Total tickets:", len(baseline_df))
print("Valid predictions:", len(valid_df))
print("Invalid predictions:", len(baseline_df) - len(valid_df))


# Classification metrics
queue_accuracy = accuracy_score(
    valid_df["true_queue"],
    valid_df["pred_queue"]
)

queue_f1 = f1_score(
    valid_df["true_queue"],
    valid_df["pred_queue"],
    average="macro",
    zero_division=0
)

priority_accuracy = accuracy_score(
    valid_df["true_priority"],
    valid_df["pred_priority"]
)

priority_f1 = f1_score(
    valid_df["true_priority"],
    valid_df["pred_priority"],
    average="macro",
    zero_division=0
)


# Joint accuracy
joint_accuracy = (
    (valid_df["true_queue"] == valid_df["pred_queue"]) &
    (valid_df["true_priority"] == valid_df["pred_priority"])
).mean()


# JSON validity across ALL tickets
json_validity = baseline_df["valid_json"].mean()


print("\n========== BASELINE RESULTS ==========")
print(f"Queue Accuracy:       {queue_accuracy:.4f}")
print(f"Queue Macro-F1:       {queue_f1:.4f}")
print(f"Priority Accuracy:    {priority_accuracy:.4f}")
print(f"Priority Macro-F1:    {priority_f1:.4f}")
print(f"Joint Accuracy:       {joint_accuracy:.4f}")
print(f"JSON Validity:        {json_validity:.4f}")

Total tickets: 2375
Valid predictions: 2277
Invalid predictions: 98

========== BASELINE RESULTS ==========
Queue Accuracy:       0.3303
Queue Macro-F1:       0.2040
Priority Accuracy:    0.3944
Priority Macro-F1:    0.2375
Joint Accuracy:       0.1590
JSON Validity:        0.9587


In [25]:
strict_joint_accuracy = (
    (baseline_df["true_queue"] == baseline_df["pred_queue"]) &
    (baseline_df["true_priority"] == baseline_df["pred_priority"])
).mean()

print(f"Strict Joint Accuracy: {strict_joint_accuracy:.4f}")

Strict Joint Accuracy: 0.1524


In [26]:
strict_queue_accuracy = (
    baseline_df["true_queue"] == baseline_df["pred_queue"]
).mean()

strict_priority_accuracy = (
    baseline_df["true_priority"] == baseline_df["pred_priority"]
).mean()

print(f"Strict Queue Accuracy:    {strict_queue_accuracy:.4f}")
print(f"Strict Priority Accuracy: {strict_priority_accuracy:.4f}")
print(f"Strict Joint Accuracy:    {strict_joint_accuracy:.4f}")

Strict Queue Accuracy:    0.3166
Strict Priority Accuracy: 0.3781
Strict Joint Accuracy:    0.1524


## 4. QLoRA configuration

The final training configuration used:

- **Base model:** Qwen/Qwen2.5-1.5B-Instruct
- **Quantization:** 4-bit NF4 with double quantization
- **Compute dtype:** FP16
- **LoRA rank:** 16
- **LoRA alpha:** 32
- **LoRA dropout:** 0.05
- **Target modules:** `q_proj`, `k_proj`, `v_proj`, `o_proj`
- **Gradient checkpointing:** enabled
- **Epochs:** 3
- **Learning rate:** 2e-4
- **Effective batch size:** 16
- **Maximum sequence length:** 512

Only about **0.2815%** of the model parameters were trainable.


In [6]:
import os

PROJECT_DIR = "/content/drive/MyDrive/SupportOpsAI"

print(os.listdir(PROJECT_DIR))

['data']


In [7]:
import pandas as pd

processed_path = f"{PROJECT_DIR}/data/processed"

train_df = pd.read_csv(f"{processed_path}/train.csv")
val_df = pd.read_csv(f"{processed_path}/validation.csv")
test_df = pd.read_csv(f"{processed_path}/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (18997, 6)
Validation: (2375, 6)
Test: (2375, 6)


In [8]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded")
print("Pad token:", tokenizer.pad_token)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded
Pad token: <|endoftext|>


In [9]:
from datasets import Dataset

SYSTEM_PROMPT = """You are a customer support ticket classifier.

Choose exactly one queue from:
Technical Support
Product Support
Customer Service
IT Support
Billing and Payments
Returns and Exchanges
Service Outages and Maintenance
Sales and Pre-Sales
Human Resources
General Inquiry

Choose exactly one priority from:
low
medium
high

Return ONLY valid JSON in exactly this format:
{"queue": "<queue>", "priority": "<priority>"}
"""


def format_ticket(row):
    subject = row["subject"] if pd.notna(row["subject"]) else ""

    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": (
                    f"Subject: {subject}\n\n"
                    f"Body: {row['body']}"
                )
            },
            {
                "role": "assistant",
                "content": (
                    f'{{"queue": "{row["queue"]}", '
                    f'"priority": "{row["priority"]}"}}'
                )
            }
        ]
    }


train_dataset = Dataset.from_list(
    train_df.apply(format_ticket, axis=1).tolist()
)

val_dataset = Dataset.from_list(
    val_df.apply(format_ticket, axis=1).tolist()
)

test_dataset = Dataset.from_list(
    test_df.apply(format_ticket, axis=1).tolist()
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['messages'],
    num_rows: 18997
})
Dataset({
    features: ['messages'],
    num_rows: 2375
})
Dataset({
    features: ['messages'],
    num_rows: 2375
})


In [10]:
def apply_chat_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return example


train_dataset = train_dataset.map(apply_chat_template)
val_dataset = val_dataset.map(apply_chat_template)
test_dataset = test_dataset.map(apply_chat_template)

Map:   0%|          | 0/18997 [00:00<?, ? examples/s]

Map:   0%|          | 0/2375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2375 [00:00<?, ? examples/s]

In [11]:
print(train_dataset[0]["text"])

<|im_start|>system
You are a customer support ticket classifier.

Choose exactly one queue from:
Technical Support
Product Support
Customer Service
IT Support
Billing and Payments
Returns and Exchanges
Service Outages and Maintenance
Sales and Pre-Sales
Human Resources
General Inquiry

Choose exactly one priority from:
low
medium
high

Return ONLY valid JSON in exactly this format:
{"queue": "<queue>", "priority": "<priority>"}
<|im_end|>
<|im_start|>user
Subject: Inquiry on Jenkins Integration with Our SaaS Platform

Body: Hello, I am contacting you to request information about integrating Jenkins with your project management software. Could you share any documentation or guides you have for this process? I am particularly interested in understanding the benefits and challenges of this integration. I look forward to your response and to discussing this further.<|im_end|>
<|im_start|>assistant
{"queue": "Product Support", "priority": "medium"}<|im_end|>



In [12]:
import numpy as np

lengths = []

for example in train_dataset:
    tokens = tokenizer(
        example["text"],
        add_special_tokens=False
    )["input_ids"]

    lengths.append(len(tokens))

print("Min:", min(lengths))
print("Max:", max(lengths))
print("Average:", np.mean(lengths))

Min: 122
Max: 438
Average: 191.51250197399588


In [42]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [43]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

OutOfMemoryError: CUDA out of memory. Tried to allocate 892.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 225.81 MiB is free. Including non-PyTorch memory, this process has 14.34 GiB memory in use. Of the allocated memory 14.04 GiB is allocated by PyTorch, and 162.38 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [15]:
from peft import LoraConfig, get_peft_model


lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
)


model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [16]:
print("Model type:", type(model))
print("Trainable parameters:")

model.print_trainable_parameters()

Model type: <class 'peft.peft_model.PeftModelForCausalLM'>
Trainable parameters:
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [37]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=output_dir,

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,

    # Learning rate
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=180,

    # T4 -> FP16
    fp16=True,
    bf16=False,

    # Faster training
    gradient_checkpointing=False,
    use_cache=True,

    # Sequence length
    max_length=512,
    packing=False,

    # Train on completion/answer tokens
    completion_only_loss=True,

    # Validation
    eval_strategy="epoch",
    per_device_eval_batch_size=8,

    # Checkpoints
    save_strategy="epoch",
    save_total_limit=2,

    # Best checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Logging
    logging_strategy="steps",
    logging_steps=50,
    report_to="none",

    seed=42,

    dataset_text_field="text",
)

## 5. Training

The final training run was completed from a Drive-backed checkpoint after earlier runtime interruptions. The notebook keeps the recorded training/resume output rather than requiring the run to be repeated.


In [5]:
train_result = trainer.train(
    resume_from_checkpoint=last_checkpoint
)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
3,0.557514,0.561491,0.551457,3638163.000000,0.847351


In [6]:
# Save the final trained LoRA adapter

final_model_path = "/content/drive/MyDrive/SupportOpsAI/models/supportopsai-qlora-final"

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print("Final adapter saved to:")
print(final_model_path)

Final adapter saved to:
/content/drive/MyDrive/SupportOpsAI/models/supportopsai-qlora-final


In [7]:
import os

print(os.listdir(final_model_path))

['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin']


## 6. Final QLoRA evaluation

The saved adapter was loaded and evaluated on the **untouched 2,375-example test set**. The final evaluation uses deterministic generation with enough output tokens to avoid truncating otherwise valid JSON responses.


In [12]:
import os
import json
import torch
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score


# ============================================================
# 1. Paths
# ============================================================

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

project_path = "/content/drive/MyDrive/SupportOpsAI"
model_path = f"{project_path}/models/supportopsai-qlora-final"
test_path = f"{project_path}/data/processed/test.csv"
results_dir = f"{project_path}/results"

os.makedirs(results_dir, exist_ok=True)


# ============================================================
# 2. Mount Google Drive
# ============================================================

from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

print("Drive ready.")


# ============================================================
# 3. Check that the trained adapter exists
# ============================================================

print("\nModel directory:")
print(model_path)

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"\nCould not find:\n{model_path}\n\n"
        "Check whether the final adapter was saved there."
    )

print("\nFiles in model directory:")
print(os.listdir(model_path))


# ============================================================
# 4. Load tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(model_path)

print("\nTokenizer loaded.")


# ============================================================
# 5. Load the base Qwen model in 4-bit
# ============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("Base model loaded.")


# ============================================================
# 6. Load our trained LoRA adapter
# ============================================================

model = PeftModel.from_pretrained(
    base_model,
    model_path,
)

model.eval()

print("LoRA adapter loaded.")
print("Model ready for evaluation.")


# ============================================================
# 7. Load untouched test set
# ============================================================

test_df = pd.read_csv(test_path)

print("\nTest set:")
print("Rows:", len(test_df))
print(test_df[["queue", "priority"]].head())


# ============================================================
# 8. Prompt
# ============================================================

SYSTEM_PROMPT = """You are a customer support ticket classification assistant.

Given a support ticket, classify it into exactly one queue and exactly one priority.

Allowed queues:
- Technical Support
- Product Support
- Customer Service
- IT Support
- Billing and Payments
- Returns and Exchanges
- Service Outages and Maintenance
- Sales and Pre-Sales
- Human Resources
- General Inquiry

Allowed priorities:
- low
- medium
- high

Return ONLY valid JSON in exactly this format:
{"queue": "<queue>", "priority": "<priority>"}
"""


# ============================================================
# 9. Prediction function
# ============================================================

def predict_ticket(subject, body):

    subject = subject if pd.notna(subject) else ""
    body = body if pd.notna(body) else ""

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"Subject: {subject}\n\nBody: {body}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,

            # IMPORTANT:
            # Previous evaluation used 30 and many outputs
            # were truncated just before the final "}".
            max_new_tokens=50,

            do_sample=False,

            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


# ============================================================
# 10. Parse prediction
# ============================================================

def parse_prediction(raw):

    try:
        prediction = json.loads(raw)

        if not isinstance(prediction, dict):
            return None

        if "queue" not in prediction or "priority" not in prediction:
            return None

        return prediction

    except Exception:
        return None


# ============================================================
# 11. Evaluate entire test set
# ============================================================

predictions = []

print("\nStarting evaluation...")
print(f"Total tickets: {len(test_df)}")
print("This may take a while on the T4.\n")

for i, row in test_df.iterrows():

    raw_output = predict_ticket(
        row["subject"],
        row["body"]
    )

    parsed = parse_prediction(raw_output)

    predictions.append({
        "index": i,
        "subject": row["subject"],
        "body": row["body"],
        "true_queue": row["queue"],
        "true_priority": row["priority"],
        "raw_output": raw_output,
        "predicted_queue": (
            parsed["queue"] if parsed else None
        ),
        "predicted_priority": (
            parsed["priority"] if parsed else None
        ),
        "valid_json": parsed is not None,
    })

    if (i + 1) % 100 == 0:
        print(f"Evaluated {i + 1}/{len(test_df)}")


pred_df = pd.DataFrame(predictions)


# ============================================================
# 12. Calculate metrics
# ============================================================

total = len(pred_df)

valid_mask = pred_df["valid_json"]

valid_df = pred_df[valid_mask].copy()

valid_count = len(valid_df)
invalid_count = total - valid_count


# ---- Valid prediction metrics ----

if valid_count > 0:

    queue_accuracy = accuracy_score(
        valid_df["true_queue"],
        valid_df["predicted_queue"]
    )

    queue_macro_f1 = f1_score(
        valid_df["true_queue"],
        valid_df["predicted_queue"],
        average="macro"
    )

    priority_accuracy = accuracy_score(
        valid_df["true_priority"],
        valid_df["predicted_priority"]
    )

    priority_macro_f1 = f1_score(
        valid_df["true_priority"],
        valid_df["predicted_priority"],
        average="macro"
    )

    joint_accuracy = (
        (valid_df["true_queue"] == valid_df["predicted_queue"]) &
        (valid_df["true_priority"] == valid_df["predicted_priority"])
    ).mean()

else:

    queue_accuracy = 0
    queue_macro_f1 = 0
    priority_accuracy = 0
    priority_macro_f1 = 0
    joint_accuracy = 0


# ---- JSON validity ----

json_validity = valid_count / total


# ---- Strict end-to-end metrics ----
# Invalid JSON counts as incorrect.

strict_queue_accuracy = (
    (pred_df["true_queue"] == pred_df["predicted_queue"]) &
    pred_df["valid_json"]
).mean()

strict_priority_accuracy = (
    (pred_df["true_priority"] == pred_df["predicted_priority"]) &
    pred_df["valid_json"]
).mean()

strict_joint_accuracy = (
    (pred_df["true_queue"] == pred_df["predicted_queue"]) &
    (pred_df["true_priority"] == pred_df["predicted_priority"]) &
    pred_df["valid_json"]
).mean()


# ============================================================
# 13. Print final results
# ============================================================

print("\n" + "=" * 60)
print("FINAL QLoRA EVALUATION")
print("=" * 60)

print(f"\nTotal tickets:       {total}")
print(f"Valid predictions:   {valid_count}")
print(f"Invalid predictions: {invalid_count}")

print("\n--- Valid Prediction Metrics ---")

print(f"Queue Accuracy:      {queue_accuracy:.4f}")
print(f"Queue Macro-F1:      {queue_macro_f1:.4f}")

print(f"Priority Accuracy:   {priority_accuracy:.4f}")
print(f"Priority Macro-F1:   {priority_macro_f1:.4f}")

print(f"Joint Accuracy:      {joint_accuracy:.4f}")

print(f"JSON Validity:       {json_validity:.4f}")

print("\n--- Strict End-to-End Metrics ---")

print(f"Strict Queue Acc:    {strict_queue_accuracy:.4f}")
print(f"Strict Priority Acc: {strict_priority_accuracy:.4f}")
print(f"Strict Joint Acc:    {strict_joint_accuracy:.4f}")


# ============================================================
# 14. Save predictions + metrics
# ============================================================

predictions_path = f"{results_dir}/qlora_test_predictions.csv"

pred_df.to_csv(
    predictions_path,
    index=False
)

metrics = {
    "total_tickets": total,
    "valid_predictions": valid_count,
    "invalid_predictions": invalid_count,

    "queue_accuracy_valid": queue_accuracy,
    "queue_macro_f1_valid": queue_macro_f1,

    "priority_accuracy_valid": priority_accuracy,
    "priority_macro_f1_valid": priority_macro_f1,

    "joint_accuracy_valid": joint_accuracy,

    "json_validity": json_validity,

    "strict_queue_accuracy": strict_queue_accuracy,
    "strict_priority_accuracy": strict_priority_accuracy,
    "strict_joint_accuracy": strict_joint_accuracy,
}

metrics_path = f"{results_dir}/qlora_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("\nSaved:")
print(predictions_path)
print(metrics_path)


# ============================================================
# 15. Show examples
# ============================================================

print("\n" + "=" * 60)
print("SAMPLE PREDICTIONS")
print("=" * 60)

for _, row in pred_df.head(10).iterrows():

    print("\nTrue:")
    print({
        "queue": row["true_queue"],
        "priority": row["true_priority"]
    })

    print("Model:")
    print(row["raw_output"])

Drive ready.

Model directory:
/content/drive/MyDrive/SupportOpsAI/models/supportopsai-qlora-final

Files in model directory:
['adapter_config.json', 'README.md', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer_config.json', 'training_args.bin', 'tokenizer.json']

Tokenizer loaded.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded.
LoRA adapter loaded.
Model ready for evaluation.

Test set:
Rows: 2375
                             queue priority
0  Service Outages and Maintenance     high
1                Technical Support   medium
2             Billing and Payments   medium
3                  Product Support     high
4                       IT Support     high

Starting evaluation...
Total tickets: 2375
This may take a while on the T4.

Evaluated 100/2375
Evaluated 200/2375
Evaluated 300/2375
Evaluated 400/2375
Evaluated 500/2375
Evaluated 600/2375
Evaluated 700/2375
Evaluated 800/2375
Evaluated 900/2375
Evaluated 1000/2375
Evaluated 1100/2375
Evaluated 1200/2375
Evaluated 1300/2375
Evaluated 1400/2375
Evaluated 1500/2375
Evaluated 1600/2375
Evaluated 1700/2375
Evaluated 1800/2375
Evaluated 1900/2375
Evaluated 2000/2375
Evaluated 2100/2375
Evaluated 2200/2375
Evaluated 2300/2375

FINAL QLoRA EVALUATION

Total tickets:       2375
Valid predictions:   2375
Invalid predictions: 0

--- Valid Predic

## 7. Error analysis

The recorded analysis below examines class distribution, per-class precision/recall/F1, confusion matrices, and the most common queue confusions.


In [13]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)
import pandas as pd
import numpy as np


# ============================================================
# 1. Queue class distribution in TEST SET
# ============================================================

print("=" * 70)
print("TRUE QUEUE DISTRIBUTION")
print("=" * 70)

queue_distribution = (
    test_df["queue"]
    .value_counts()
    .rename_axis("queue")
    .reset_index(name="count")
)

queue_distribution["percentage"] = (
    queue_distribution["count"] / len(test_df) * 100
).round(2)

print(queue_distribution.to_string(index=False))


# ============================================================
# 2. Queue classification report
# ============================================================

valid_df = pred_df[pred_df["valid_json"]].copy()

queue_labels = sorted(test_df["queue"].unique())

print("\n" + "=" * 70)
print("QUEUE CLASSIFICATION REPORT")
print("=" * 70)

queue_report = classification_report(
    valid_df["true_queue"],
    valid_df["predicted_queue"],
    labels=queue_labels,
    output_dict=True,
    zero_division=0
)

queue_report_df = pd.DataFrame(queue_report).T

print(
    queue_report_df[
        ["precision", "recall", "f1-score", "support"]
    ].round(3).to_string()
)


# ============================================================
# 3. Queue confusion matrix
# ============================================================

print("\n" + "=" * 70)
print("QUEUE CONFUSION MATRIX")
print("=" * 70)

queue_cm = confusion_matrix(
    valid_df["true_queue"],
    valid_df["predicted_queue"],
    labels=queue_labels
)

queue_cm_df = pd.DataFrame(
    queue_cm,
    index=queue_labels,
    columns=queue_labels
)

print("\nRows = TRUE")
print("Columns = PREDICTED\n")

print(queue_cm_df.to_string())


# ============================================================
# 4. Most common queue mistakes
# ============================================================

mistakes = valid_df[
    valid_df["true_queue"] != valid_df["predicted_queue"]
].copy()

mistake_pairs = (
    mistakes
    .groupby(["true_queue", "predicted_queue"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("\n" + "=" * 70)
print("TOP QUEUE CONFUSIONS")
print("=" * 70)

print(
    mistake_pairs.head(20).to_string(index=False)
)


# ============================================================
# 5. What does the model predict?
# ============================================================

print("\n" + "=" * 70)
print("MODEL PREDICTED QUEUE DISTRIBUTION")
print("=" * 70)

pred_queue_distribution = (
    valid_df["predicted_queue"]
    .value_counts()
    .rename_axis("queue")
    .reset_index(name="predicted_count")
)

print(
    pred_queue_distribution.to_string(index=False)
)


# ============================================================
# 6. Priority classification report
# ============================================================

priority_labels = sorted(test_df["priority"].unique())

print("\n" + "=" * 70)
print("PRIORITY CLASSIFICATION REPORT")
print("=" * 70)

priority_report = classification_report(
    valid_df["true_priority"],
    valid_df["predicted_priority"],
    labels=priority_labels,
    output_dict=True,
    zero_division=0
)

priority_report_df = pd.DataFrame(priority_report).T

print(
    priority_report_df[
        ["precision", "recall", "f1-score", "support"]
    ].round(3).to_string()
)


# ============================================================
# 7. Priority confusion matrix
# ============================================================

print("\n" + "=" * 70)
print("PRIORITY CONFUSION MATRIX")
print("=" * 70)

priority_cm = confusion_matrix(
    valid_df["true_priority"],
    valid_df["predicted_priority"],
    labels=priority_labels
)

priority_cm_df = pd.DataFrame(
    priority_cm,
    index=priority_labels,
    columns=priority_labels
)

print("\nRows = TRUE")
print("Columns = PREDICTED\n")

print(priority_cm_df.to_string())

TRUE QUEUE DISTRIBUTION
                          queue  count  percentage
              Technical Support    686       28.88
                Product Support    444       18.69
               Customer Service    357       15.03
                     IT Support    283       11.92
           Billing and Payments    242       10.19
          Returns and Exchanges    117        4.93
Service Outages and Maintenance     94        3.96
            Sales and Pre-Sales     72        3.03
                Human Resources     46        1.94
                General Inquiry     34        1.43

QUEUE CLASSIFICATION REPORT
                                 precision  recall  f1-score  support
Billing and Payments                 0.971   0.545     0.698   242.00
Customer Service                     0.238   0.042     0.071   357.00
General Inquiry                      0.000   0.000     0.000    34.00
Human Resources                      1.000   0.022     0.043    46.00
IT Support                          

## 8. Final results

| Metric | Baseline | QLoRA |
|---|---:|---:|
| Queue Accuracy | 31.66% | **36.97%** |
| Queue Macro-F1 | **20.40%** | 19.41% |
| Priority Accuracy | 37.81% | **47.54%** |
| Priority Macro-F1 | 23.75% | **35.27%** |
| Joint Accuracy | 15.24% | **21.05%** |
| JSON Validity | 95.87% | **100.00%** |

### Interpretation

QLoRA improved priority classification, joint accuracy, and queue accuracy relative to the base model. Queue macro-F1 decreased slightly, and the error analysis shows a strong prediction bias toward **Technical Support**, so queue classification remains a limitation of the current experiment.


In [14]:
# ============================================================
# FINAL RESULTS — SupportOpsAI
# ============================================================

import os
import json
import pandas as pd

results_dir = "/content/drive/MyDrive/SupportOpsAI/results"
os.makedirs(results_dir, exist_ok=True)

# Final results from the completed evaluation
baseline_results = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "test_samples": 2375,
    "valid_predictions": 2277,
    "invalid_predictions": 98,
    "queue_accuracy": 0.3166,
    "queue_macro_f1": 0.2040,
    "priority_accuracy": 0.3781,
    "priority_macro_f1": 0.2375,
    "joint_accuracy": 0.1524,
    "json_validity": 0.9587
}

qlora_results = {
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
    "method": "QLoRA",
    "test_samples": 2375,
    "valid_predictions": 2375,
    "invalid_predictions": 0,
    "queue_accuracy": 0.3697,
    "queue_macro_f1": 0.1941,
    "priority_accuracy": 0.4754,
    "priority_macro_f1": 0.3527,
    "joint_accuracy": 0.2105,
    "json_validity": 1.0000
}

# Save individual result files
with open(os.path.join(results_dir, "baseline_results.json"), "w") as f:
    json.dump(baseline_results, f, indent=4)

with open(os.path.join(results_dir, "qlora_results.json"), "w") as f:
    json.dump(qlora_results, f, indent=4)


# Comparison table
comparison = pd.DataFrame({
    "Metric": [
        "Queue Accuracy",
        "Queue Macro-F1",
        "Priority Accuracy",
        "Priority Macro-F1",
        "Joint Accuracy",
        "JSON Validity"
    ],
    "Baseline": [
        0.3166,
        0.2040,
        0.3781,
        0.2375,
        0.1524,
        0.9587
    ],
    "QLoRA": [
        0.3697,
        0.1941,
        0.4754,
        0.3527,
        0.2105,
        1.0000
    ]
})

comparison["Change"] = comparison["QLoRA"] - comparison["Baseline"]

comparison.to_csv(
    os.path.join(results_dir, "model_comparison.csv"),
    index=False
)

print("Results saved to:")
print(results_dir)

print("\nFiles:")
print("- baseline_results.json")
print("- qlora_results.json")
print("- model_comparison.csv")

Results saved to:
/content/drive/MyDrive/SupportOpsAI/results

Files:
- baseline_results.json
- qlora_results.json
- model_comparison.csv


# Final Evaluation

The fine-tuned Qwen2.5-1.5B-Instruct model was evaluated on the untouched
2,375-example test set and compared against the base model.

| Metric | Baseline | QLoRA |
|---|---:|---:|
| Queue Accuracy | 31.66% | **36.97%** |
| Queue Macro-F1 | **20.40%** | 19.41% |
| Priority Accuracy | 37.81% | **47.54%** |
| Priority Macro-F1 | 23.75% | **35.27%** |
| Joint Accuracy | 15.24% | **21.05%** |
| JSON Validity | 95.87% | **100.00%** |

## Key Findings

- QLoRA improved queue accuracy from 31.66% to 36.97%.
- Priority accuracy improved from 37.81% to 47.54%.
- Priority Macro-F1 improved from 23.75% to 35.27%.
- Joint accuracy improved from 15.24% to 21.05%.
- JSON validity reached 100% after using sufficient generation length.
- Queue classification remained challenging, with the model showing a strong
  bias toward the Technical Support class.

In [16]:
# ============================================================
# SAVE SUPPORTOPSAI EVALUATION ARTIFACTS TO GOOGLE DRIVE
# ============================================================

import os
import json

results_dir = "/content/drive/MyDrive/SupportOpsAI/results"
os.makedirs(results_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. Save full prediction dataframe
# ------------------------------------------------------------

predictions_path = os.path.join(
    results_dir,
    "qlora_test_predictions.csv"
)

pred_df.to_csv(predictions_path, index=False)


# ------------------------------------------------------------
# 2. Save queue distribution
# ------------------------------------------------------------

queue_distribution.to_csv(
    os.path.join(results_dir, "test_queue_distribution.csv"),
    index=False
)


# ------------------------------------------------------------
# 3. Save queue classification report
# ------------------------------------------------------------

queue_report_df.to_csv(
    os.path.join(results_dir, "queue_classification_report.csv")
)


# ------------------------------------------------------------
# 4. Save queue confusion matrix
# ------------------------------------------------------------

queue_cm_df.to_csv(
    os.path.join(results_dir, "queue_confusion_matrix.csv")
)


# ------------------------------------------------------------
# 5. Save top queue mistakes
# ------------------------------------------------------------

mistake_pairs.to_csv(
    os.path.join(results_dir, "queue_mistake_pairs.csv"),
    index=False
)


# ------------------------------------------------------------
# 6. Save predicted queue distribution
# ------------------------------------------------------------

pred_queue_distribution.to_csv(
    os.path.join(results_dir, "predicted_queue_distribution.csv"),
    index=False
)


# ------------------------------------------------------------
# 7. Save priority classification report
# ------------------------------------------------------------

priority_report_df.to_csv(
    os.path.join(results_dir, "priority_classification_report.csv")
)


# ------------------------------------------------------------
# 8. Save priority confusion matrix
# ------------------------------------------------------------

priority_cm_df.to_csv(
    os.path.join(results_dir, "priority_confusion_matrix.csv")
)


# ------------------------------------------------------------
# 9. Save final headline metrics
# ------------------------------------------------------------

final_metrics = {
    "test_samples": 2375,
    "valid_predictions": 2375,
    "invalid_predictions": 0,

    "queue_accuracy": 0.3697,
    "queue_macro_f1": 0.1941,

    "priority_accuracy": 0.4754,
    "priority_macro_f1": 0.3527,

    "joint_accuracy": 0.2105,

    "json_validity": 1.0000
}

with open(
    os.path.join(results_dir, "qlora_final_metrics.json"),
    "w"
) as f:
    json.dump(final_metrics, f, indent=4)


# ------------------------------------------------------------
# 10. Save baseline vs QLoRA comparison
# ------------------------------------------------------------

comparison_df = pd.DataFrame({
    "Metric": [
        "Queue Accuracy",
        "Queue Macro-F1",
        "Priority Accuracy",
        "Priority Macro-F1",
        "Joint Accuracy",
        "JSON Validity"
    ],

    "Baseline": [
        0.3166,
        0.2040,
        0.3781,
        0.2375,
        0.1524,
        0.9587
    ],

    "QLoRA": [
        0.3697,
        0.1941,
        0.4754,
        0.3527,
        0.2105,
        1.0000
    ]
})

comparison_df["Absolute Change"] = (
    comparison_df["QLoRA"] -
    comparison_df["Baseline"]
)

comparison_df.to_csv(
    os.path.join(results_dir, "baseline_vs_qlora.csv"),
    index=False
)


# ------------------------------------------------------------
# 11. Save a human-readable summary
# ------------------------------------------------------------

summary = """SupportOpsAI — Final QLoRA Evaluation
=======================================

Model:
Qwen/Qwen2.5-1.5B-Instruct

Method:
4-bit QLoRA

Test set:
2,375 untouched test examples

Results:
Queue Accuracy:       36.97%
Queue Macro-F1:       19.41%
Priority Accuracy:    47.54%
Priority Macro-F1:    35.27%
Joint Accuracy:       21.05%
JSON Validity:       100.00%

Baseline → QLoRA:
Queue Accuracy:       31.66% → 36.97%
Queue Macro-F1:       20.40% → 19.41%
Priority Accuracy:    37.81% → 47.54%
Priority Macro-F1:    23.75% → 35.27%
Joint Accuracy:       15.24% → 21.05%
JSON Validity:        95.87% → 100.00%

Main observation:
The QLoRA model substantially improved priority classification and
joint accuracy. Queue classification remained challenging, with
a strong prediction bias toward the Technical Support class.
"""

with open(
    os.path.join(results_dir, "README_results.txt"),
    "w"
) as f:
    f.write(summary)


# ------------------------------------------------------------
# DONE
# ------------------------------------------------------------

print("=" * 60)
print("EVALUATION ARTIFACTS SAVED")
print("=" * 60)
print(f"\nLocation:\n{results_dir}\n")

for filename in sorted(os.listdir(results_dir)):
    print("✓", filename)

EVALUATION ARTIFACTS SAVED

Location:
/content/drive/MyDrive/SupportOpsAI/results

✓ README_results.txt
✓ baseline_results.json
✓ baseline_vs_qlora.csv
✓ model_comparison.csv
✓ predicted_queue_distribution.csv
✓ priority_classification_report.csv
✓ priority_confusion_matrix.csv
✓ qlora_final_metrics.json
✓ qlora_metrics.json
✓ qlora_results.json
✓ qlora_test_predictions.csv
✓ queue_classification_report.csv
✓ queue_confusion_matrix.csv
✓ queue_mistake_pairs.csv
✓ test_queue_distribution.csv
